# 02 · Regresión múltiple aplicada: Ames Housing

**Módulo 3 · Sesión 6** — Regresión lineal

## Objetivos

El notebook 01 verificó, sobre datos sintéticos, que el descenso del gradiente implementado
a mano llega al mismo resultado que la ecuación normal. Aquí se deja el descenso manual de
lado —`scikit-learn` ya lo resuelve internamente— y el foco pasa a lo que importa en la
práctica: ajustar regresión múltiple sobre un dataset real dentro de un `Pipeline`
(módulo 2), calcular todas las métricas de `01-regresion-lineal.md` e **interpretar los
residuales** para verificar los supuestos del modelo.

**Dataset conductor del módulo:** Ames Housing — 2930 viviendas vendidas en Ames, Iowa
(2006-2010), 80 variables. Predecimos `saleprice`.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42

## 1. Datos y variables

El módulo 2 ya cubrió limpieza y diagnóstico de calidad a fondo (sobre Titanic); no se
repite aquí. Se seleccionan 15 variables con menos del 1 % de nulos, con criterio de
dominio: tamaño de la vivienda, calidad, antigüedad y tipo de construcción — los factores
que cualquier tasador consideraría primero.

In [ ]:
datos = pd.read_csv("../datos/ames-housing.csv")

numericas = [
    "gr_liv_area",      # área habitable sobre el nivel del suelo (pies²)
    "total_bsmt_sf",    # área de sótano
    "garage_area",
    "garage_cars",
    "overall_qual",     # calidad general, 1-10
    "overall_cond",     # condición general, 1-10
    "year_built",
    "year_remod_add",   # año de la última remodelación
    "lot_area",
    "full_bath",
    "bedroom_abvgr",
]
categoricas = ["bldg_type", "house_style", "central_air"]

X = datos[numericas + categoricas]
y = datos["saleprice"]

print(f"X: {X.shape}   nulos totales: {X.isnull().sum().sum()}")
X.describe().T[["mean", "std", "min", "max"]].round(1)

## 2. Partición y línea base

Antes de ajustar nada, la referencia trivial: predecir siempre el precio promedio del
entrenamiento (la misma idea de `01-primer-modelo-aplicado.ipynb`, módulo 1). Cualquier
modelo que no supere esto no vale el esfuerzo de entrenarlo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)

prediccion_base = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
rmse_base = mean_squared_error(y_test, prediccion_base) ** 0.5
print(f"RMSE de la línea base (predecir el promedio): ${rmse_base:,.0f}")

## 3. Pipeline: `ColumnTransformer` + regresión lineal

Igual que en `04-pipeline-caracteristicas-aplicado.ipynb` (módulo 2): todo el preprocesamiento
—imputación, escalado, codificación— vive dentro del `Pipeline`, ajustado solo con
`X_train`, para no repetir ninguna fuga de las medidas en el notebook 03 de ese módulo.

In [ ]:
def crear_modelo():
    """Devuelve un Pipeline nuevo, con su propio preprocesador.

    Es una función y no un objeto reutilizable a propósito: `Pipeline` **no clona** los pasos
    que recibe, así que si dos pipelines compartieran el mismo `ColumnTransformer`, el último
    `fit` dejaría al otro con un preprocesador ajustado sobre datos que no le corresponden.
    """
    transformador = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputar", SimpleImputer(strategy="median")),
                    ("escalar", StandardScaler()),
                ]),
                numericas,
            ),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", drop="first"),
                categoricas,
            ),
        ]
    )
    return Pipeline([
        ("preprocesar", transformador),
        ("regresor", LinearRegression()),
    ])


modelo = crear_modelo()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

## 4. Métricas

Las cinco de `01-regresion-lineal.md`, sobre el conjunto de prueba.

In [ ]:
n_test = len(y_test)
p = modelo.named_steps["preprocesar"].transform(X_train).shape[1]  # columnas tras codificar

mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
r2_ajustado = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)

metricas = pd.DataFrame(
    {
        "métrica": ["RMSE línea base", "MSE", "RMSE", "MAE", "R²", "R² ajustado"],
        "valor": [rmse_base, mse, rmse, mae, r2, r2_ajustado],
    }
)
metricas["valor"] = metricas["valor"].round(3)
metricas

El modelo baja el RMSE muy por debajo de la línea base, y $R^2$ y $R^2$ ajustado casi no se
separan — con $n=2930$ y $p \approx 20$, la penalización de `01-regresion-lineal.md` apenas
se nota. Esa brecha se vuelve importante cuando $p$ crece mucho respecto a $n$ (sesión 8).

## 5. Residuales: ¿se cumplen los supuestos?

`01-regresion-lineal.md` señaló la homocedasticidad como el supuesto que más se ignora y
más fácil es de ver. Se grafica.

In [ ]:
residuales = y_test.to_numpy() - y_pred

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))

ejes[0].scatter(y_pred, residuales, alpha=0.3, s=15, edgecolor="none")
ejes[0].axhline(0, color="black", linewidth=1)
ejes[0].set_xlabel("Valor ajustado ($)")
ejes[0].set_ylabel("Residual ($)")
ejes[0].set_title("Residuales vs. ajustados")

ejes[1].hist(residuales, bins=40, edgecolor="white")
ejes[1].set_xlabel("Residual ($)")
ejes[1].set_title("Distribución de residuales")

plt.tight_layout()
plt.show()

El gráfico de la izquierda tiene forma de embudo: la dispersión de los residuales crece con
el precio. El modelo se equivoca poco en viviendas baratas y mucho —en dólares absolutos—
en las caras. Es una violación clara de homocedasticidad (supuesto 3), y el histograma
confirma una cola derecha: unos pocos errores muy grandes, casi todos sobre-predicciones o
sub-predicciones de viviendas caras.

## 6. ¿Ayuda predecir en escala logarítmica?

`saleprice` está sesgada a la derecha (pocas viviendas muy caras). Una transformación común
es predecir $\log(1 + \text{precio})$ en vez del precio directo, y deshacer la
transformación al final con $\exp(\cdot) - 1$. La hipótesis: en log, un error proporcional
(10 % de más o de menos) pesa igual en una vivienda barata que en una cara, así que el
ajuste debería dejar de concentrarse en las caras. Se mide, en vez de asumirlo.

In [ ]:
y_train_log = np.log1p(y_train)

modelo_log = crear_modelo()
modelo_log.fit(X_train, y_train_log)

y_pred_log = modelo_log.predict(X_test)
y_pred_dolares = np.expm1(y_pred_log)  # de vuelta a dólares, para comparar en las mismas unidades

residuales_log = y_test.to_numpy() - y_pred_dolares
rmse_log = mean_squared_error(y_test, y_pred_dolares) ** 0.5
mae_log = mean_absolute_error(y_test, y_pred_dolares)

comparacion_log = pd.DataFrame(
    {
        "métrica": ["RMSE ($)", "MAE ($)", "MAPE (%)"],
        "sobre precio": [rmse, mae, 100 * np.mean(np.abs(residuales) / y_test)],
        "sobre log(precio)": [
            rmse_log,
            mae_log,
            100 * np.mean(np.abs(residuales_log) / y_test),
        ],
    }
)
comparacion_log.round(1)

El resultado es mixto, y por eso vale la pena reportarlo en vez de un solo número: **MAE y
MAPE mejoran** con el modelo en log —el error típico baja— pero el **RMSE empeora**. La
razón aparece al mirar el peor error de cada modelo.

In [ ]:
peor_precio = np.abs(residuales).max()
idx_peor_log = np.argmax(np.abs(residuales_log))
peor_log = np.abs(residuales_log[idx_peor_log])

print(f"Peor error absoluto — modelo sobre precio:      ${peor_precio:,.0f}")
print(f"Peor error absoluto — modelo sobre log(precio): ${peor_log:,.0f}")
print(
    f"  → vivienda real: ${y_test.iloc[idx_peor_log]:,.0f}, "
    f"predicha en ${y_pred_dolares[idx_peor_log]:,.0f}"
)

Una sola vivienda —grande, de calidad máxima (`overall_qual` = 10), pero vendida muy por
debajo de lo que sus características sugieren— produce un error de más de 700 mil dólares
en el modelo logarítmico al revertir la transformación, casi el doble del peor error del
modelo directo. En escala log el modelo la predice razonablemente bien *en términos
relativos*; pero como $\exp(\cdot)$ es convexa, un error moderado en log se convierte en un
error enorme en dólares para una vivienda cara. El RMSE, que eleva al cuadrado, es
extremadamente sensible a ese único punto.

Si la culpa del RMSE es de un caso extremo, entonces el resto de las viviendas debería estar
mejor predicho por el modelo en log. Se comprueba partiendo el conjunto de prueba en
cuartiles de precio.

In [ ]:
cuartil = pd.qcut(y_test, 4, labels=["Q1 (barato)", "Q2", "Q3", "Q4 (caro)"])

por_cuartil = (
    pd.DataFrame(
        {
            "cuartil": cuartil.to_numpy(),
            "MAE sobre precio": np.abs(residuales),
            "MAE sobre log(precio)": np.abs(residuales_log),
        }
    )
    .groupby("cuartil", observed=True)
    .mean()
)
por_cuartil.round(0)

El modelo en log gana en **los cuatro cuartiles**, incluido el de las viviendas más caras:
el error típico es más bajo en cada tramo del precio. O sea que el RMSE global no empeora
porque el modelo sea peor en general, sino por el único caso extremo de la celda anterior.

Esta es la lección, no un tecnicismo: **RMSE y MAE pueden discrepar sobre cuál modelo es
mejor**, y la transformación del objetivo cambia qué errores penaliza el ajuste. Elegir la
métrica —y la escala de predicción— es una decisión de negocio (¿importa más el error
típico o evitar un desastre puntual?), no un paso mecánico. Se retoma en la sesión 8.

## 7. Coeficientes del modelo en log(precio)

Con el objetivo en escala log, cada coeficiente $\beta_j$ se interpreta aproximadamente como
"un incremento de una desviación estándar en $x_j$ cambia el precio en $(\exp(\beta_j)-1)
\times 100\,\%$", manteniendo lo demás constante.

In [ ]:
nombres = modelo_log.named_steps["preprocesar"].get_feature_names_out()
coeficientes = modelo_log.named_steps["regresor"].coef_

tabla_coef = pd.DataFrame({"variable": nombres, "beta": coeficientes})
tabla_coef["cambio_%_aprox"] = (np.exp(tabla_coef["beta"]) - 1) * 100
tabla_coef.sort_values("beta", ascending=False).head(8).round(3)

`overall_qual` (calidad general) es, con diferencia, el coeficiente más grande entre las
variables numéricas — consistente con lo que un tasador diría de memoria. Varias de las
variables de tamaño (`gr_liv_area`, `total_bsmt_sf`, `garage_area`, `garage_cars`) miden
aspectos parecidos de "qué tan grande es la casa": es exactamente el tipo de solapamiento
que la sesión 7 diagnostica con VIF antes de decidir si regularizar.

## Resumen

| Resultado | Conecta con |
|---|---|
| RMSE del modelo muy por debajo de la línea base; $R^2$ y $R^2$ ajustado casi iguales | `01-regresion-lineal.md`; la brecha crecerá cuando $p/n$ aumente (sesión 8) |
| Residuales en embudo: heterocedasticidad clara | Supuesto 3 de `01-regresion-lineal.md` |
| Modelar $\log(1+\text{precio})$ baja MAE y MAPE, pero **sube** el RMSE por un solo caso extremo | RMSE vs. MAE no siempre coinciden; elegir métrica es una decisión, no un trámite (sesión 8) |
| Varias variables de tamaño con coeficientes grandes y relacionados entre sí | Multicolinealidad y VIF (sesión 7, siguiente notebook) |